In [1]:
import caf.base as cb
import caf.tem as ct
import pandas as pd
import os
from pathlib import Path

This tests specific inputs/ outputs run locally using python-refactor branch of NTS-Processing (pre-caf.nts).
HB Production and HB Attraction match, except for normits zone 5248009 - this zone has two tfn_at's associated with it, which is where we think the discrepancy stems from.

# HB Production
## Preprocessing
### Create equivalent population DVector

Nhan's code writes a .csv called pop_2023.csv\
It comes from landuse Output P11, the code translates and adds aws to the segmentation, amongst other actions.

Create DVector from Nhan's Output P11 derived pop_2023.csv...\
zoning: lsoa_2021\
segmentation: [gender_3, aws, ns_sec, soc, accom_hh]

In [2]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\lu_pop_2023.hdf"
if not os.path.exists(dvec_path):
    pop = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\lu_pop_2023.csv")
    pop = pop.set_index(["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"])#.drop(columns=["adult_nssec"])
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"],
                                            naming_order=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("lsoa_2021")
    pop_dvec = cb.DVector(segmentation=segmentation, import_data=pop, zoning_system=zoning_system)
    pop_dvec.save(dvec_path)

### Create equivalent trip rates DVector

In [3]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\hb_trip_rates_production.hdf"
if not os.path.exists(dvec_path):
    tr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\hb_trip_rates_production.csv")
    tr = tr.rename(columns={"gender": "gender_3", "ns": "ns_sec", "purpose": "p"}).pivot(index=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"], columns="tfn_at", values="beta")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"],
                                                        naming_order=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    tr_dvec = cb.DVector(segmentation=segmentation, import_data=tr, zoning_system=zoning_system)
    tr_dvec.save(dvec_path)

### Create equivalent 2023 adjustment DVector

In [4]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\trip_rate_adjustments_production_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create equivalent mts DVector

In [5]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\mode_time_split_production_hb_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input\mode_time_split_production_hb_fr_reg.csv")
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp", "hh_type"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp", "hh_type"],
                                                        naming_order=["p", "m", "tp", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

## TEM Setup

In [6]:
tem = ct.TEM(
    model_years=[2023],
    scenario="Core",
    output_zoning="normits",
    iteration_name="final_reporting",
    export_home=r"T:\ThomasPrince\TEM Input\comparison\caf.tem output",
    return_segmentation=["hh_type", "p", "m", "tp"]
)

## HB Production Model Setup and Run

In [7]:
input_dir = Path(r"T:\ThomasPrince\TEM Input\comparison\01_HBProduction\input")

HBProd = tem.HBProductionModel(
    population_paths={2023: input_dir / "lu_pop_2023.hdf"},
    trip_rates_path=input_dir / "hb_trip_rates_production.hdf",
    mode_time_splits_path=input_dir / "mode_time_split_production_hb_fr_reg.hdf",
    adjustment_path=input_dir / "trip_rate_adjustments_production_hb_fr.hdf",
    population_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_pop.csv"
)

In [8]:
HBProd.run()

C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\segmentation.py:342: SegmentationWarning: Read in level p is a subset of the segment. If this was not expected check the input segmentation.
  warnings.warn(
C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:179: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1001001' '1001002' '1001003' ... '11358007' '11358008' '11358009']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  zones.loc[:, name] = zones[name].astype(str)
C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\caf.base\src\caf\base\zoning.py:681: TranslationWarning: 20 tfn_at zones have splitting factors which don't sum to 1 (value totals may change during zone_translation), the maximum difference is 7.7e+02
  warnings.warn(
C:\Users\Spiral\Documents\Thomas Princ

In [ ]:
check = cb.DVector.load(HBProd.model.export_paths.tem_segmented[2023])
check.aggregate(["p"]).data

In [ ]:
pop_2023 = pd.read_csv(r"C:\Users\Spiral\Documents\Thomas Prince\Common Analytical Framework\NTS-Processing_python-refactor\NoTEM\voa_gb_2023_uni\reports\pop_2023_normits.csv")
pop_2023 = pop_2023.groupby(["normits_v3.3_id"])[["1","2","3","4","5","6","7","8"]].sum().T
pop_2023.index = pop_2023.index.astype(int)
pop_2023

In [ ]:
test = (check.aggregate(["p"]).data - pop_2023).stack()
test = test.reset_index()
test = test.groupby("normits_id")[0].sum()
test.loc[abs(test)>1] # Where the difference in total trips, by normits id, is > 1

There's one normits zone to look into...\
5248009\
NB. this normits zone has two tfn_at's - this is where the error will stem from...

# HB Attraction
## Preprocessing
### Create equivalent trip rates DVectors

In [12]:
tr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_attraction.csv")
tr = tr.loc[tr["dir"]=="hb"]

# Segmentations
soc = cb.Segmentation(cb.SegmentationInput(enum_segments=["soc"], naming_order=["soc"]))
sic = cb.Segmentation(cb.SegmentationInput(enum_segments=["sic_2_digit"], naming_order=["sic_2_digit"]))
total = cb.Segmentation(cb.SegmentationInput(enum_segments=["total"], naming_order=["total"]))
tfn_at = cb.ZoningSystem.get_zoning("tfn_at")

#p1
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf"):
    tr1 = tr.loc[tr["p"]==1]
    tr1 = tr1.pivot(index = "soc", columns="tfn_at", values="alpha")
    tr1.index = tr1.index.astype(int)
    cb.DVector(segmentation=soc, import_data=tr1, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf")

#p2
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf"):
    tr2 = tr.loc[tr["p"]==2]
    tr2 = tr2.pivot(index = "soc", columns="tfn_at", values="alpha")
    tr2.index = tr2.index.astype(int)
    cb.DVector(segmentation=soc, import_data=tr2, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf")

#p3
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf"):
    tr3 = tr.loc[tr["p"]==3]
    tr3["sic_2_digit"] = 85
    tr3 = tr3.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr3, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf")

#p4
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf"):
    tr4 = tr.loc[tr["p"]==4]
    tr4["sic_2_digit"] = tr4.loc[:, "e_code"].apply(lambda x: [46, 47])
    tr4 = tr4.explode("sic_2_digit")
    tr4 = tr4.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr4, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf")

#p5
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf"):
    tr5 = tr.loc[tr["p"]==5]
    tr5_1 = tr5.loc[tr5["e_code"]=="e08"].copy()
    tr5_1["sic_2_digit"] = 86
    tr5_2 = tr5.loc[tr5["e_code"]=="e09"].copy()
    tr5_2["sic_2_digit"] = tr5_2.loc[:, "e_code"].apply(lambda x: [64, 65, 66, 68, 69, 75, 77, 79, 80, 95, 96])
    tr5_2 = tr5_2.explode("sic_2_digit")
    tr5_3 = tr5.loc[tr5["e_code"]=="e11"].copy()
    tr5_3["sic_2_digit"] = 56
    tr5 = pd.concat([tr5_1,tr5_2,tr5_3])
    tr5 = tr5.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr5, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf")

#p6
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf"):
    tr6 = tr.loc[tr["p"]==6]
    tr6["sic_2_digit"] = tr6["e_code"].apply(lambda x: [90, 91, 92, 93, 94])
    tr6 = tr6.explode("sic_2_digit")
    tr6 = tr6.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr6, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf")

#p7
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf"):
    tr7 = tr.loc[tr["p"]==7]
    tr7["total"] = 1
    tr7 = tr7.pivot(index = "total", columns="tfn_at", values="alpha")
    tr7.index = tr7.index.astype(int)
    cb.DVector(segmentation=total, import_data=tr7, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf")

#p8
if not os.path.exists(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf"):
    tr8 = tr.loc[tr["p"]==8]
    tr8["sic_2_digit"] = tr8.loc[:, "e_code"].apply(lambda x: [2, 3, 55])
    tr8 = tr8.explode("sic_2_digit")
    tr8 = tr8.pivot(index = "sic_2_digit", columns="tfn_at", values="alpha")
    cb.DVector(segmentation=sic, import_data=tr8, zoning_system=tfn_at).save(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf")


### Create Equivalent trip rates adjustment

In [13]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments_attractions_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create Equivalent MTS DVec

In [14]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.csv")
    mts = mts.loc[mts["uni"]==0]
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp"],
                                                        naming_order=["p", "m", "tp"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

### Create Equivalent MTS Adjustment

In [15]:
dvec_path = r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p", "period": "tp", "mode": "m"}).loc[(adj["pa"]=="a") & (adj["direction"]=="hb_fr")].pivot(index=["p", "tp", "m"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "tp", "m"],
                                                        naming_order=["p", "tp", "m"])) # TODO delete and rewrite adj with "a"
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

## HB Attraction Model setup

In [16]:
HBAttr = tem.HBAttractionModel(
    trip_rates_paths={
        1: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p1.hdf",
        2: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p2.hdf",
        3: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p3.hdf",
        4: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p4.hdf",
        5: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p5.hdf",
        6: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p6.hdf",
        7: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p7.hdf",
        8: r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rates_p8.hdf"
    },
    balance_production=True,
    emp_landuse_paths = {2023: r"F:\Deliverables\Land-Use\241213_Employment\02_Final Outputs\Output E6.hdf"},
    hh_landuse_dirs = {2023: r"F:\Deliverables\Land-Use\241220_Populationv2\02_Final Outputs"},
    hh_landuse_prefix = "Output P13.3",
    mode_time_splits_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_attraction_hb_fr_reg.hdf",
    trip_rate_adjustment_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\trip_rate_adjustments_attractions_hb_fr.hdf",
    hh_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_pop.csv",
    emp_translation_path=r"T:\ThomasPrince\TEM Input\lsoa_normits_emp.csv",
    mode_time_splits_adjustment_path=r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\input\mode_time_split_adjustments.hdf"
)

In [ ]:
HBAttr.run()

In [28]:
check = cb.DVector.load(HBAttr.model.export_paths.mts_demand_adj[2023])
mdl_attr = pd.read_csv(r"T:\ThomasPrince\TEM Input\comparison\02_HBAttraction\mdlnotem_outputs\emp_2023_normits.csv")
mdl_attr = mdl_attr.set_index("normits_v3.3_id")[["1","2","3","4","5","6","7","8"]].T
mdl_attr.index = mdl_attr.index.astype(int)

In [ ]:
check.aggregate(["p"]).data#.sum()

In [ ]:
mdl_attr#.sum().sum()

In [ ]:
(check.data / mdl_attr).stack().describe()

In [24]:
test = (check.data / mdl_attr).stack()
test.loc[abs(test)>1.00001]

Again, it is only this one zone where any difference in trips occur

In [ ]:
End of testing.